In [1]:

import sys
import subprocess
from pathlib import Path
import re
import copy
import shutil
import json

# Install dependencies in the current Colab/runtime.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "openpyxl", "gradio"])

import pandas as pd
import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter
import gradio as gr

# Mount Google Drive when running in Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Drive mount skipped: {exc}")

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/data")
DEFAULT_RAW_FILE = DATA_DIR / "2026-04-raw-transactions.xlsx"
INITIAL_TEMPLATE_WORKBOOK = DATA_DIR / "2026-03-workbook.xlsx"

EXPECTED_TABS = [
    "00_Summary",
    "01_Account_Summary",
    "02_Department_Summary",
    "03_Counterparty_Summary",
    "04_Review_Items",
    "05_Detail",
    "06_Notes",
]

YELLOW_FILL = PatternFill(fill_type="solid", fgColor="FFF59D")


def find_existing_variant(path_like):
    """
    Finds the exact file first, then looks for reasonable duplicate-download variants such as
    '2026-03-workbook (1).xlsx' or '2026-03-workbook(2).xlsx'.
    """
    path = Path(path_like)
    if path.exists():
        return path

    stem = path.stem
    suffix = path.suffix or ".xlsx"
    candidates = sorted(path.parent.glob(f"{stem}*{suffix}"))
    for candidate in candidates:
        if candidate.name.startswith(stem) and candidate.suffix.lower() == suffix.lower():
            return candidate
    return path


def extract_month_identifier(file_path, df=None):
    file_path = Path(file_path)
    match = re.search(r"(20\d{2}-\d{2})", file_path.name)
    if match:
        return match.group(1)

    if df is not None and "Txn Date" in df.columns and df["Txn Date"].notna().any():
        return pd.to_datetime(df["Txn Date"]).min().strftime("%Y-%m")

    raise ValueError(
        "Could not determine the month identifier. Please include YYYY-MM in the file name "
        "or make sure the Txn Date column is populated."
    )


def prior_month_identifier(month_id):
    return (pd.Period(month_id, freq="M") - 1).strftime("%Y-%m")


def resolve_prior_workbook(raw_path, data_dir, fallback_template=None, df=None):
    month_id = extract_month_identifier(raw_path, df=df)
    expected_prior = Path(data_dir) / f"{prior_month_identifier(month_id)}-workbook.xlsx"
    resolved_prior = find_existing_variant(expected_prior)

    warnings = []
    if resolved_prior.exists():
        return resolved_prior, warnings

    if fallback_template is not None:
        fallback_template = find_existing_variant(fallback_template)
        if fallback_template.exists():
            warnings.append(f"Expected prior workbook not found: {expected_prior.name}")
            warnings.append(f"Falling back to template workbook: {fallback_template.name}")
            return fallback_template, warnings

    raise FileNotFoundError(
        f"No usable prior workbook was found. Expected: {expected_prior.name}"
    )


def copy_row_style(ws, source_row, target_row, max_col):
    for col_idx in range(1, max_col + 1):
        src = ws.cell(source_row, col_idx)
        dst = ws.cell(target_row, col_idx)
        if src.has_style:
            dst._style = copy.copy(src._style)
        dst.font = copy.copy(src.font)
        dst.fill = copy.copy(src.fill)
        dst.border = copy.copy(src.border)
        dst.alignment = copy.copy(src.alignment)
        dst.protection = copy.copy(src.protection)
        dst.number_format = src.number_format
    ws.row_dimensions[target_row].height = ws.row_dimensions[source_row].height
    ws.row_dimensions[target_row].hidden = ws.row_dimensions[source_row].hidden


def ensure_capacity(ws, start_row, needed_rows, template_row, max_col):
    current_capacity = max(ws.max_row - start_row + 1, 0)
    if needed_rows > current_capacity:
        extra_rows = needed_rows - current_capacity
        insert_at = ws.max_row + 1
        ws.insert_rows(insert_at, amount=extra_rows)
        for row_idx in range(insert_at, insert_at + extra_rows):
            copy_row_style(ws, template_row, row_idx, max_col)


def clear_body(ws, start_row, max_col):
    for row_idx in range(start_row, ws.max_row + 1):
        for col_idx in range(1, max_col + 1):
            ws.cell(row_idx, col_idx).value = None


def find_raw_sheet(raw_wb, expected_headers):
    for ws in raw_wb.worksheets:
        headers = [ws.cell(1, col_idx).value for col_idx in range(1, len(expected_headers) + 1)]
        if headers == expected_headers:
            return ws

    for ws in raw_wb.worksheets:
        if ws.max_column >= len(expected_headers):
            return ws

    raise ValueError("No usable raw transactions sheet was found.")


def summarize_dataframe(df, category_col):
    grouped = (
        df.assign(**{"Signed Amount": df["Amount"].where(df["Flow"].eq("Inflow"), -df["Amount"])})
          .groupby(category_col, dropna=False, as_index=False)["Signed Amount"]
          .sum()
          .rename(columns={category_col: "Category", "Signed Amount": "Net Amount"})
    )
    grouped["Category"] = grouped["Category"].fillna("(blank)")
    return grouped


def write_two_col_summary(ws, df_out, sort_mode):
    start_row = 2
    max_col = 2
    ensure_capacity(ws, start_row, len(df_out), template_row=2, max_col=max_col)
    clear_body(ws, start_row, max_col)

    if sort_mode == "alpha":
        df_out = df_out.sort_values(["Category"], kind="stable").reset_index(drop=True)
    elif sort_mode == "net_desc":
        df_out = df_out.sort_values(
            ["Net Amount", "Category"],
            ascending=[False, True],
            kind="stable"
        ).reset_index(drop=True)

    for row_idx, record in enumerate(df_out.to_dict("records"), start=start_row):
        ws.cell(row_idx, 1).value = record["Category"]
        ws.cell(row_idx, 2).value = float(round(record["Net Amount"], 2))

    ws.auto_filter.ref = f"A1:B{start_row + len(df_out) - 1}" if len(df_out) else "A1:B1"
    ws.freeze_panes = "A2"


def write_table_sheet(ws, df_out):
    start_row = 2
    max_col = df_out.shape[1]
    ensure_capacity(ws, start_row, len(df_out), template_row=2, max_col=max_col)
    clear_body(ws, start_row, max_col)

    for row_idx, row in enumerate(df_out.itertuples(index=False), start=start_row):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row_idx, col_idx).value = value

    last_row = start_row + len(df_out) - 1 if len(df_out) else 1
    ws.auto_filter.ref = f"A1:{get_column_letter(max_col)}{last_row}"
    ws.freeze_panes = "A2"


def inspect_workbooks(prior_workbook_path, raw_file_path):
    prior_workbook_path = find_existing_variant(prior_workbook_path)
    raw_file_path = find_existing_variant(raw_file_path)

    if not prior_workbook_path.exists():
        raise FileNotFoundError(f"Prior workbook not found: {prior_workbook_path}")
    if not raw_file_path.exists():
        raise FileNotFoundError(f"Raw transactions file not found: {raw_file_path}")

    prior_wb = load_workbook(prior_workbook_path, data_only=True)
    raw_wb = load_workbook(raw_file_path, data_only=True)

    report = {
        "prior_workbook": str(prior_workbook_path),
        "raw_file": str(raw_file_path),
        "prior_tabs": {},
        "raw_tabs": {},
        "structure_match": None,
        "discrepancies": [],
    }

    for ws in prior_wb.worksheets:
        first_row = [ws.cell(1, col_idx).value for col_idx in range(1, ws.max_column + 1)]
        report["prior_tabs"][ws.title] = {
            "rows": ws.max_row,
            "cols": ws.max_column,
            "first_row": first_row,
        }

    for ws in raw_wb.worksheets:
        first_row = [ws.cell(1, col_idx).value for col_idx in range(1, ws.max_column + 1)]
        report["raw_tabs"][ws.title] = {
            "rows": ws.max_row,
            "cols": ws.max_column,
            "first_row": first_row,
        }

    detail_headers = [
        prior_wb["05_Detail"].cell(1, col_idx).value
        for col_idx in range(1, prior_wb["05_Detail"].max_column + 1)
    ]
    raw_ws = find_raw_sheet(raw_wb, detail_headers)
    raw_headers = [raw_ws.cell(1, col_idx).value for col_idx in range(1, len(detail_headers) + 1)]

    report["structure_match"] = raw_headers == detail_headers
    if raw_headers != detail_headers:
        for col_idx, (expected, actual) in enumerate(zip(detail_headers, raw_headers), start=1):
            if expected != actual:
                report["discrepancies"].append(
                    f"Column {col_idx}: expected '{expected}', found '{actual}'"
                )

    raw_df = pd.read_excel(raw_file_path, sheet_name=raw_ws.title)
    raw_row_count = len(raw_df)
    if raw_row_count != 220:
        report["discrepancies"].append(
            f"Reference text says 220 transactions, but the uploaded raw file contains {raw_row_count} data rows."
        )

    prior_counterparty_filter = prior_wb["03_Counterparty_Summary"].auto_filter.ref
    prior_counterparty_max_row = prior_wb["03_Counterparty_Summary"].max_row
    if prior_counterparty_filter != f"A1:B{prior_counterparty_max_row}":
        report["discrepancies"].append(
            "03_Counterparty_Summary in the prior workbook has a filter range that does not cover all populated rows. "
            "The generator will normalize it in the output."
        )

    return report


def print_inspection_report(report):
    print("=" * 88)
    print("STEP 1 INSPECTION REPORT")
    print("=" * 88)
    print(f"Prior workbook: {report['prior_workbook']}")
    print(f"Raw file      : {report['raw_file']}")
    print()

    print("Prior workbook tabs and layout")
    for tab_name, meta in report["prior_tabs"].items():
        print(f"- {tab_name}: rows={meta['rows']}, cols={meta['cols']}, first_row={meta['first_row']}")

    print()
    print("Raw workbook tabs and layout")
    for tab_name, meta in report["raw_tabs"].items():
        print(f"- {tab_name}: rows={meta['rows']}, cols={meta['cols']}, first_row={meta['first_row']}")

    print()
    print("Inferred processing logic")
    print("- 00_Summary: top-line metrics (Total Inflow, Total Outflow, Net Change, Transactions, Items Flagged for Review).")
    print("- 01_Account_Summary: group Detail rows by Account, signed net amount, alphabetical sort.")
    print("- 02_Department_Summary: group Detail rows by Department, signed net amount, alphabetical sort.")
    print("- 03_Counterparty_Summary: group Detail rows by Counterparty, signed net amount, sorted by Net Amount descending.")
    print("- 04_Review_Items: copy only rows where Needs Review = 'Yes'.")
    print("- 05_Detail: copy all raw transaction rows in the original 14-column order.")
    print("- 06_Notes: workbook notes and generation warnings; no aggregation logic.")
    print()

    print(f"Detail-column structure match: {report['structure_match']}")
    if report["discrepancies"]:
        print("Discrepancies found:")
        for item in report["discrepancies"]:
            print(f"  * {item}")
    else:
        print("Discrepancies found: none")
    print("=" * 88)
    print()


def generate_workbook(raw_path, data_dir=DATA_DIR, fallback_template=INITIAL_TEMPLATE_WORKBOOK):
    raw_path = Path(raw_path)
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    archived_raw_path = data_dir / raw_path.name
    if raw_path.resolve() != archived_raw_path.resolve():
        shutil.copy2(raw_path, archived_raw_path)
    else:
        archived_raw_path = raw_path

    preview_df = pd.read_excel(archived_raw_path, sheet_name=0)
    prior_workbook_path, resolver_warnings = resolve_prior_workbook(
        archived_raw_path,
        data_dir=data_dir,
        fallback_template=fallback_template,
        df=preview_df,
    )

    prior_wb_for_headers = load_workbook(prior_workbook_path)
    if prior_wb_for_headers.sheetnames != EXPECTED_TABS:
        raise ValueError(
            f"Prior workbook tabs do not match the expected structure: {prior_wb_for_headers.sheetnames}"
        )

    detail_headers = [
        prior_wb_for_headers["05_Detail"].cell(1, col_idx).value
        for col_idx in range(1, 15)
    ]

    raw_wb = load_workbook(archived_raw_path, data_only=True)
    raw_ws = find_raw_sheet(raw_wb, detail_headers)
    raw_headers = [raw_ws.cell(1, col_idx).value for col_idx in range(1, 15)]
    if raw_headers != detail_headers:
        mismatch_lines = []
        for col_idx, (expected, actual) in enumerate(zip(detail_headers, raw_headers), start=1):
            if expected != actual:
                mismatch_lines.append(
                    f"Column {col_idx}: expected '{expected}', got '{actual}'"
                )
        raise ValueError(
            "Raw file columns do not match the prior workbook Detail tab.\n" + "\n".join(mismatch_lines)
        )

    df = pd.read_excel(archived_raw_path, sheet_name=raw_ws.title)
    df["Txn Date"] = pd.to_datetime(df["Txn Date"], errors="coerce")
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

    warnings = list(resolver_warnings)

    invalid_flow_mask = ~df["Flow"].isin(["Inflow", "Outflow"])
    if invalid_flow_mask.any():
        warnings.append(f"{invalid_flow_mask.sum()} row(s) have an unexpected Flow value.")

    if df["Amount"].isna().any():
        warnings.append(f"{df['Amount'].isna().sum()} row(s) have a blank or non-numeric Amount.")

    required_fields = ["Txn ID", "Txn Date", "Flow", "Account", "Department", "Counterparty", "Amount"]
    missing_required = {
        column: int(df[column].isna().sum())
        for column in required_fields
        if df[column].isna().any()
    }
    if missing_required:
        warnings.append(
            "Missing required values: " + ", ".join([f"{key}={value}" for key, value in missing_required.items()])
        )

    month_id = extract_month_identifier(archived_raw_path, df=df)
    output_path = data_dir / f"{month_id}-workbook.xlsx"

    shutil.copy2(prior_workbook_path, output_path)
    wb = load_workbook(output_path)

    total_inflow = float(df.loc[df["Flow"].eq("Inflow"), "Amount"].sum())
    total_outflow = float(df.loc[df["Flow"].eq("Outflow"), "Amount"].sum())
    net_change = float(total_inflow - total_outflow)
    review_df = df[df["Needs Review"].astype(str).str.strip().str.lower().eq("yes")].copy()

    # 00_Summary
    summary_ws = wb["00_Summary"]
    summary_ws["A2"] = f"Monthly Management Workbook - {month_id}"
    summary_ws["B5"] = total_inflow
    summary_ws["B6"] = total_outflow
    summary_ws["B7"] = net_change
    summary_ws["B8"] = int(len(df))
    summary_ws["B9"] = int(len(review_df))
    summary_ws.auto_filter.ref = "A4:B9"

    # 01 to 03 summary tabs
    account_df = summarize_dataframe(df, "Account")
    department_df = summarize_dataframe(df, "Department")
    counterparty_df = summarize_dataframe(df, "Counterparty")

    write_two_col_summary(wb["01_Account_Summary"], account_df, sort_mode="alpha")
    write_two_col_summary(wb["02_Department_Summary"], department_df, sort_mode="alpha")
    write_two_col_summary(wb["03_Counterparty_Summary"], counterparty_df, sort_mode="net_desc")

    # 04 and 05 detail-style tabs
    ordered_cols = detail_headers
    detail_df = df[ordered_cols].copy()
    write_table_sheet(wb["05_Detail"], detail_df)
    write_table_sheet(wb["04_Review_Items"], review_df[ordered_cols].copy())

    # Highlight unclear or problematic source values
    flow_col = ordered_cols.index("Flow") + 1
    amount_col = ordered_cols.index("Amount") + 1

    for row_idx, is_invalid in enumerate(invalid_flow_mask.tolist(), start=2):
        if is_invalid:
            wb["05_Detail"].cell(row_idx, flow_col).fill = copy.copy(YELLOW_FILL)

    for row_idx, amount_value in enumerate(df["Amount"].tolist(), start=2):
        if pd.isna(amount_value):
            wb["05_Detail"].cell(row_idx, amount_col).fill = copy.copy(YELLOW_FILL)

    review_index = review_df.index.tolist()
    for review_row_idx, raw_idx in zip(range(2, 2 + len(review_df)), review_index):
        if invalid_flow_mask.loc[raw_idx]:
            wb["04_Review_Items"].cell(review_row_idx, flow_col).fill = copy.copy(YELLOW_FILL)
        if pd.isna(df.loc[raw_idx, "Amount"]):
            wb["04_Review_Items"].cell(review_row_idx, amount_col).fill = copy.copy(YELLOW_FILL)

    # 06_Notes
    notes_ws = wb["06_Notes"]
    for row_idx in range(1, max(notes_ws.max_row, 50) + 1):
        notes_ws.cell(row_idx, 1).value = None
        notes_ws.cell(row_idx, 1).fill = copy.copy(notes_ws.cell(1, 1).fill)

    notes_lines = [
        "Workbook Notes",
        "",
        f"Generated from raw file: {archived_raw_path.name}",
        f"Prior workbook used: {Path(prior_workbook_path).name}",
        f"Output month identifier: {month_id}",
        "Aggregation logic: signed net amount = Inflow positive, Outflow negative.",
        "01_Account_Summary and 02_Department_Summary are sorted alphabetically by Category.",
        "03_Counterparty_Summary is sorted by Net Amount descending to match the prior workbook style.",
        "04_Review_Items contains only rows where Needs Review = 'Yes' (case-insensitive).",
        f"Source rows loaded from the raw sheet: {len(df)}.",
    ]

    if warnings:
        notes_lines.append("Warnings:")
        notes_lines.extend([f"- {warning}" for warning in warnings])
    else:
        notes_lines.append("Warnings: none.")

    for row_idx, line in enumerate(notes_lines, start=1):
        notes_ws.cell(row_idx, 1).value = line

    for row_idx in range(1, len(notes_lines) + 1):
        text_value = str(notes_ws.cell(row_idx, 1).value or "")
        if text_value.startswith("Warnings") or text_value.startswith("-"):
            notes_ws.cell(row_idx, 1).fill = copy.copy(YELLOW_FILL)

    notes_ws.column_dimensions["A"].width = 120

    wb.save(output_path)

    return {
        "output_path": str(output_path),
        "rows_loaded": int(len(df)),
        "review_rows": int(len(review_df)),
        "total_inflow": round(total_inflow, 2),
        "total_outflow": round(total_outflow, 2),
        "net_change": round(net_change, 2),
        "warnings": warnings,
        "prior_workbook_used": str(prior_workbook_path),
    }


def validate_output(output_path):
    output_path = Path(output_path)
    wb = load_workbook(output_path, data_only=True)
    detail_df = pd.read_excel(output_path, sheet_name="05_Detail")
    review_df = pd.read_excel(output_path, sheet_name="04_Review_Items")
    account_df = pd.read_excel(output_path, sheet_name="01_Account_Summary")
    department_df = pd.read_excel(output_path, sheet_name="02_Department_Summary")
    counterparty_df = pd.read_excel(output_path, sheet_name="03_Counterparty_Summary")

    signed_amount = detail_df["Amount"].where(detail_df["Flow"].eq("Inflow"), -detail_df["Amount"])

    def diff_count(key, summary_df):
        calc = (
            pd.DataFrame({"Category": detail_df[key], "Net Amount": signed_amount})
            .groupby("Category", as_index=False)["Net Amount"]
            .sum()
        )
        merged = calc.merge(summary_df, on="Category", how="outer", suffixes=("_calc", "_sheet"))
        merged["diff"] = (merged["Net Amount_calc"].fillna(0) - merged["Net Amount_sheet"].fillna(0)).round(2)
        return int((merged["diff"] != 0).sum())

    validation = {
        "tabs_match_expected": wb.sheetnames == EXPECTED_TABS,
        "detail_rows": int(len(detail_df)),
        "review_rows": int(len(review_df)),
        "review_only_yes": bool(
            set(review_df["Needs Review"].astype(str).str.strip().str.lower().unique()) <= {"yes"}
        ),
        "account_diff_rows": diff_count("Account", account_df),
        "department_diff_rows": diff_count("Department", department_df),
        "counterparty_diff_rows": diff_count("Counterparty", counterparty_df),
        "summary_inflow": round(float(detail_df.loc[detail_df["Flow"].eq("Inflow"), "Amount"].sum()), 2),
        "summary_outflow": round(float(detail_df.loc[detail_df["Flow"].eq("Outflow"), "Amount"].sum()), 2),
    }
    validation["net_change"] = round(validation["summary_inflow"] - validation["summary_outflow"], 2)
    return validation


def gradio_generate(raw_file):
    if raw_file is None:
        return "Please upload a raw transactions file.", None

    uploaded_path = Path(raw_file)
    result = generate_workbook(uploaded_path)
    validation = validate_output(result["output_path"])

    lines = [
        f"Workbook created: {Path(result['output_path']).name}",
        f"Prior workbook used: {Path(result['prior_workbook_used']).name}",
        f"Rows loaded: {result['rows_loaded']}",
        f"Review rows: {result['review_rows']}",
        f"Total inflow: {result['total_inflow']:,.2f}",
        f"Total outflow: {result['total_outflow']:,.2f}",
        f"Net change: {result['net_change']:,.2f}",
        f"Validation - tabs match expected: {validation['tabs_match_expected']}",
        f"Validation - review only yes: {validation['review_only_yes']}",
        f"Validation - aggregation diffs: account={validation['account_diff_rows']}, "
        f"department={validation['department_diff_rows']}, counterparty={validation['counterparty_diff_rows']}",
    ]

    if result["warnings"]:
        lines.append("Warnings:")
        lines.extend([f"- {warning}" for warning in result["warnings"]])

    return "\n".join(lines), result["output_path"]


# ---- Step 1 inspection on the known April test pair ----
if find_existing_variant(INITIAL_TEMPLATE_WORKBOOK).exists() and find_existing_variant(DEFAULT_RAW_FILE).exists():
    inspection = inspect_workbooks(INITIAL_TEMPLATE_WORKBOOK, DEFAULT_RAW_FILE)
    print_inspection_report(inspection)

    print("Running smoke test with the provided April file...")
    smoke_test = generate_workbook(DEFAULT_RAW_FILE)
    smoke_validation = validate_output(smoke_test["output_path"])
    print("Smoke test result:")
    print(json.dumps(smoke_test, indent=2))
    print("Smoke test validation:")
    print(json.dumps(smoke_validation, indent=2))
    print()
else:
    print("Initial test files were not found. The interface will still load for manual upload.")
    print(f"Expected prior workbook: {INITIAL_TEMPLATE_WORKBOOK}")
    print(f"Expected raw file      : {DEFAULT_RAW_FILE}")
    print()

# ---- Step 2/3 reusable Gradio interface ----
with gr.Blocks() as demo:
    gr.Markdown("# Monthly Workbook Generator")
    gr.Markdown(
        "Upload the current-month raw transactions file. The app will find the prior month workbook "
        "from the same folder, preserve the workbook format, and save the new output back to Google Drive."
    )

    raw_upload = gr.File(label="Current-month raw transactions file", type="filepath")
    run_button = gr.Button("Generate workbook")
    status_box = gr.Textbox(label="Status", lines=14)
    output_file = gr.File(label="Generated workbook")

    run_button.click(
        fn=gradio_generate,
        inputs=raw_upload,
        outputs=[status_box, output_file],
    )

launch_info = demo.launch(share=False, debug=True, prevent_thread_lock=True)
print(f"Gradio is running at: {getattr(demo, 'local_url', 'unavailable')}")


Mounted at /content/drive
STEP 1 INSPECTION REPORT
Prior workbook: /content/drive/My Drive/Colab Notebooks/data/2026-03-workbook.xlsx
Raw file      : /content/drive/My Drive/Colab Notebooks/data/2026-04-raw-transactions.xlsx

Prior workbook tabs and layout
- 00_Summary: rows=9, cols=2, first_row=['Maple Crest Distribution', None]
- 01_Account_Summary: rows=24, cols=2, first_row=['Category', 'Net Amount']
- 02_Department_Summary: rows=7, cols=2, first_row=['Category', 'Net Amount']
- 03_Counterparty_Summary: rows=48, cols=2, first_row=['Category', 'Net Amount']
- 04_Review_Items: rows=45, cols=14, first_row=['Txn ID', 'Txn Date', 'Flow', 'Account', 'Department', 'Counterparty', 'Description', 'Amount', 'Payment Method', 'Tax Code', 'Source Doc ID', 'Needs Review', 'Prepared By', 'Memo']
- 05_Detail: rows=201, cols=14, first_row=['Txn ID', 'Txn Date', 'Flow', 'Account', 'Department', 'Counterparty', 'Description', 'Amount', 'Payment Method', 'Tax Code', 'Source Doc ID', 'Needs Review', '

<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
Gradio is running at: http://127.0.0.1:7860/
